<a href="https://colab.research.google.com/github/AnaghaTiwari/anaghatiwari/blob/gh-pages/current_events_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -U langchain-community
# !pip install datasets
# !pip install faiss-cpu
!pip install pypdf

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from datasets import load_dataset

from google.colab import drive
drive.mount('/content/drive')
DB_FAISS_PATH = "/content/drive/MyDrive/faiss_db"
DATA_PATH = "/content/"

def CreateVectorDB():
    loader = DirectoryLoader(DATA_PATH,glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
                                                   chunk_overlap=50)
    texts = text_splitter.split_documents(documents)
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                       model_kwargs={'device': 'cpu'})
    db =FAISS.from_documents(texts,embeddings)
    db.save_local(DB_FAISS_PATH)

CreateVectorDB()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 5.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# !pip3 install streamlit chainlit
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 51.0 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok


In [ ]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [ ]:
!pip install ctransformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 60.9 MB/s eta 0:00:00


In [ ]:
# %%writefile app.py

from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain import PromptTemplate
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import CTransformers
from langchain.chains import RetrievalQA
import chainlit as cl

DB_FAISS_PATH = '/content/drive/MyDrive/faiss_db'

custom_prompt_template = """You are a financial analyst! Only state facts based on the news.
You can say things in terms of markets and economic analysis, in terms of risk and volatility!

Context: {context}
Question: {question}

Only return helpful answers. Be factual! Analyze everything in terms of risk, market movement, future economic outlook, market shocks and factors.
"""

def SetCustomPrompt():
    """
    Prompt template for QA retrieval for each vectorstore
    """
    prompt = PromptTemplate(template=custom_prompt_template,
                            input_variables=['context', 'question'])
    return prompt

#Retrieval QA Chain
def RetrievalQAChain(llm, prompt, db):
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=db.as_retriever(search_kwargs={'k': 2}),
                                           return_source_documents=True, chain_type_kwargs={'prompt': prompt})
    return qa_chain

def LoadLLM():
    llm = CTransformers(
        model = "TheBloke/Llama-2-7B-Chat-GGML",
        model_type="llama",
        max_new_tokens = 512,
        temperature = 0.5
    )
    return llm

def LLamaQABot():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                       model_kwargs={'device': 'cpu'})
    db = FAISS.load_local(DB_FAISS_PATH,embeddings, allow_dangerous_deserialization=True)
    llm = LoadLLM()
    qaPrompt = SetCustomPrompt()

    return RetrievalQAChain(llm,qaPrompt,db)

def GetAnswer(query):
    qaBot = LLamaQABot()
    response = qaBot.invoke({'query':query})

    return response


@cl.on_chat_start
async def Start():
    chain=LLamaQABot()

    cl.user_session.set("chain", chain)
    message = cl.Message(content="Starting Agora...")
    await message.send()
    message.content = "Hi, Welcome to Agora. How may I help you?"
    await message.update()


@cl.on_message
async def main(message):
    chain = cl.user_session.get("chain")
    langchainCallBack = cl.AsyncLangchainCallbackHandler(stream_final_answer=True, answer_prefix_tokens=["FINAL", "ANSWER"])

    langchainCallBack.answer_reached=True
    response = await chain.acall({"query": message.content})
    answer = response["result"]


    await cl.Message(content=answer).send()


GetAnswer("Hello")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

{'query': 'Hello',
 'result': "\nDo you have any thoughts on Firm A's stock price?\n",
 'source_documents': [Document(id='812d54bb-3c16-4690-8300-47c994d2326b', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-12-19T15:27:37+00:00', 'author': 'Azamat Abdymomunov, Zheng Duan, Anne Lundgaard Hansen and Ulas Misirli', 'keywords': '', 'moddate': '2024-12-30T10:15:48-05:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': 'Designing Market Shock Scenarios', 'trapped': '/False', 'source': '/content/wp24-17.pdf', 'total_pages': 38, 'page': 1, 'page_label': '2'}, page_content='Dushyanth Krishnamurthy, Tyler Davis, and Yuji Sakurai for their suggestions and ideas. We also\nthank Michael Gordy and Pawel Szerszen for their constructive feedback and suggestions.\n1'),
  Document(id='d4db806e-a830-4e28-a66c-8a87d0160e56', metadata={'producer': 'pdfTeX-1.40.25', 'creato

In [ ]:
!chainlit run app.py &>/content/logs.txt &

In [ ]:
!ngrok config add-authtoken 2tq1rxUcyHUWRpC43kSPFnJvpu1_2w5i81yLTcvjyyKJmthzp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
ngrok.kill()

In [ ]:
from pyngrok import ngrok
ngrok_tunnel = ngrok.connect(8000)
print('Public URL:', ngrok_tunnel.public_url)

Public URL: https://20fb-34-133-251-142.ngrok-free.app


In [ ]:
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Open a public URL for port 8000
public_url = ngrok.connect(port=8000)
print(f"Your Chainlit app is available at: {public_url}")

!chainlit run app.py --port 8000

ERROR:pyngrok.process.ngrok:t=2025-03-04T05:04:54+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-03-04T05:04:54+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-03-04T05:04:54+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
import os
from pyngrok import ngrok

# Define Streamlit script
streamlit_script = """
import streamlit as st
def main():
    st.set_page_config(page_title="Chat with Mellon", page_icon=":briefcase:")

    if "conversation" not in st.session_state:
        st.session_state.conversation = None

    st.header("Chat with Mellon")
    user_question = st.text_input("Ask a question about job details:")

    if user_question:
        if st.session_state.conversation:
            handle_userinput(user_question)
        else:
            st.warning("Please load and process documents first.")


    chain = cl.user_session.get("chain")
    langchainCallBack = cl.AsyncLangchainCallbackHandler(stream_final_answer=True, answer_prefix_tokens=["FINAL", "ANSWER"])

    langchainCallBack.answer_reached=True
    response = await chain.acall(message, callbacks=[langchainCallBack])
    answer = response["result"]
    sources = response["source_documents"]

    if sources:
        answer += f"\nSources:" + str(sources)
    else:
        answer += "\nNo sources found"

    await cl.Message(content=answer).send()
if __name__ == "__main__":
    main()
"""

# Save to a Python file
with open("app.py", "w") as f:
    f.write(streamlit_script)

# Start Streamlit in the background
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com
